<a href="https://colab.research.google.com/github/RENSHUZHE/HKU-DQMC/blob/main/DQMC_Theory_3_Numerical_stability_and_sign_problem.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Since each update of the Green's function is cumulatively updated based on the original Green's function, and needs to be propagated across different imaginary times, the accumulation of numerical errors will become larger and larger. The calculation of the Green's function involves matrix operations, and the rate of numerical error growth is related to the norm of the $B$ matrix. The norm of a matrix refers to the ratio of the maximum singular value to the minimum singular value. For non-interacting systems, the norm of the $B$ matrix is $\exp(\beta W)$, where $W$ is the difference between the highest and lowest energy of the energy band, i.e., the bandwidth. For interacting systems, generally speaking, the larger $\beta$ is, the stronger the interaction is, and the larger the norm of the $B$ matrix is, which we have verified in numerical calculations.

When $U$ is large or $\beta$ is very large, calculating the Green's function will have severe numerical stability problems. The reason is... the norms/condition numbers of the respective matrices are too large; when matrices are multiplied, large and small scales are mixed together. We have many algorithms that have been proven to be weakly backward stable.

Solving for the Green's function is equivalent to solving a system of linear equations:
$$\left(I_{n}+B_{L} \cdots B_{2} B_{1}\right) x=b$$
We define the condition number as the product of the norm of the matrix and the norm of its inverse:
$$\kappa\left(I_{n}+B_{L} \cdots B_{2} B_{1}\right) \equiv\left\|I_{n}+B_{L} \cdots B_{2} B_{1}\right\| \|\left(I_{n}+B_{L} \cdots B_{2} B_{1}\right)^{-1}\|$$
This quantity is roughly related to the ratio of the matrix's maximum singular value to its minimum singular value, and is related to the upper bound of the matrix multiplication error.

Because the condition number is too large, the relative error of the solution we obtain:
$$\frac{\|\tilde{x}-x\|}{\|x\|}=\mathcal{O}\left(\epsilon_{\mathrm{m}} \kappa\left(I_{n}+B_{L} \cdots B_{2} B_{1}\right)\right)$$
will be very large. Where $\epsilon_{\mathrm{m}}$ is the machine's unit round-off error. For IEEE single precision, $\epsilon_{\mathrm{m}}=2^{-24}$, and for IEEE double precision, it is $2^{-53}$. In some cases, $\epsilon_{\mathrm{m}} \kappa\left(I_{n}+B_{L} \cdots B_{2} B_{1}\right)$ easily exceeds 1, which means your solution might not have a single significant digit.

The specific operations for numerical stabilization are as follows.

First, recall the definition:
$$\mathbf{B}^{\sigma}\left(l_{2} \Delta \tau, l_{1} \Delta \tau\right)=\prod_{l=l_{1}+1}^{l_{2}} e^{\pm \alpha \operatorname{Diag}\left(\vec{S}_{l}\right)} e^{-\Delta \tau T}$$
Where $\pm$ corresponds to spin up and down, $\cosh (\alpha)=e^{\Delta \tau U / 2}$, $\operatorname{Diag}\left(\vec{S}_{l}\right)$ represents a matrix whose diagonal elements are $s_{i, l}=\pm 1$ respectively, and $T$ is a matrix that only has matrix elements of $-t$ at nearest neighbours.

To avoid numerical stability problems, we might as well perform numerical stabilization every $\tau_1$ interval. The selection of $\tau_1$ ensures that $B_{C}\left(n \tau_{1},(n-1) \tau_{1}\right)$ does not exceed machine precision. And $B_{C}(\tau, 0)=B_{C}\left(n \tau_{1},(n-1) \tau_{1}\right) \cdots B_{C}\left(\tau_{1}, 0\right)$. The first step of our numerical stabilization is to obtain the corresponding UDV matrices for $B_{C}(\tau, 0)$ or $B_{C}(\beta,\tau)$ (note, this is not directly performing singular value decomposition on these matrices). Taking $n=3$ as an example:
$$B_{C}\left(2 \tau_{1}, \tau_{1}\right) \underbrace{B_{C}\left(\tau_{1}, 0\right)}_{U_{1} D_{1} V_{1}}=\underbrace{\left(\left(B_{C}\left(2 \tau_{1}, \tau_{1}\right) U_{1}\right) D_{1}\right)}_{U_{2} D_{2} V} V_{1}=U_{2} D_{2} V_{2}$$
$$B_{C}\left(3 \tau_{1}, 2\tau_{1}\right) \underbrace{ B_{C}\left(2 \tau_{1}, 0\right) }_{U_{2} D_{2} V_{2}}=\underbrace{\left(\left(B_{C}\left(3 \tau_{1}, 2\tau_{1}\right) U_{2}\right) D_{2}\right)}_{U_{3} D_{3} V^{\prime}} V_{2}=U_{3} D_{3} V_{3}$$
Through this method, we can obtain $B_{C}(\tau, 0)=U_{R} D_{R} V_{R}$. Similarly, for $B_{C}(\beta, \tau)$, we also have $B_{C}(\beta, \tau)=V_{L} D_{L} U_{L}$ (note that earlier it was UDV, now it is VDU; earlier it was obtained by gradually doing singular value decomposition from right to left, now it is from left to right).

In practice, one often chooses to do a modified Gram-Schmidt (MGS). Where $U$ is orthogonal, $D$ is diagonal, and $V$ is an upper triangular matrix with unit diagonal elements. In short, our goal is to hope that the properties of $U$ and $V$ are very good (such matrices have very good matrix multiplication properties), and only the $D$ matrix has a large condition number. Or QR decomposition / LQ decomposition.

In fact, if stepwise SVD is not performed, or if SVD implemented by certain algorithms is used, one of the signs of numerical instability can be seen: the individual $D$ matrices containing singular values obtained from $B(\tau,0)$, that is, the corresponding singular values, will exhibit strange phenomena when $\tau$ is very large, as shown in the figure ([arXiv:2003.05286](https://arxiv.org/abs/2003.05286v6)), namely, the smaller singular values will rise and their numerical values will become identical (the figure shows the results of numerical stabilization using MATLAB's built-in svd function for $U=6, L=6, \Delta \tau=0.01$):

![Singular value stabilization graph](https://quantummc.xyz/wp-content/uploads/2023/03/WechatIMG1837-300x217.jpeg)

Based on this, there are numerous methods to calculate the Green's function:

Method 1

Given:
$$\left(\begin{array}{cc} A & B \\ C & D \end{array}\right)^{-1}=\left(\begin{array}{cc} \left(A-B D^{-1} C\right)^{-1} &\left(C-D B^{-1} A\right)^{-1} \\ \left(B-A C^{-1} D\right)^{-1} &\left(D-C A^{-1} B\right)^{-1} \end{array}\right)$$
And we have:
$$\begin{array}{c} \left(\begin{array}{cc} I & B_{C}(\beta, \tau) \\ -B_{C}(\tau, 0) & I \end{array}\right)^{-1}= \left(\begin{array}{cc} G_{C}(0,0) & G_{C}(0,\tau) \\ G_{C}(\tau,0) & G_{C}(\tau,\tau) \end{array}\right) \end{array}$$
Then:
$$\begin{aligned} &\left(\begin{array}{cc} I & V_{L} D_{L} U_{L} \\ -U_{R} D_{R} V_{R} & I \end{array}\right)^{-1}=\\ &\left[\left(\begin{array}{cc} V_{L} & 0 \\ 0 & U_{R} \end{array}\right) \underbrace{\left(\begin{array}{cc} \left(V_{R} V_{L}\right)^{-1} & D_{L} \\ -D_{R} & \left(U_{L} U_{R}\right)^{-1} \end{array}\right)}_{U D V}\left(\begin{array}{cc} V_{R} & 0 \\ 0 & U_{L} \end{array}\right)\right]^{-1}=\\ &\left[\left(\begin{array}{cc} \left(V_{R}\right)^{-1} & 0 \\ 0 & \left(U_{L}\right)^{-1} \end{array}\right) V^{-1}\right] D^{-1}\left[U^{-1}\left(\begin{array}{cc} \left(V_{L}\right)^{-1} & 0 \\ 0 & \left(U_{R}\right)^{-1} \end{array}\right)\right] \end{aligned}$$
The bottom right corner of the resulting matrix is $G(\tau,\tau)$.


Method 2
$$\begin{aligned} G(\tau,\tau)=&\left(1+B_{C}(\tau, 0) B_{C}(\beta, \tau)\right)^{-1}\\ =&\left(1+U_{R} D_{R} V_{R} V_{L} D_{L} U_{L}\right)^{-1} \\ =&\left(U_{L}\right)^{-1} \underbrace{\left(\left(U_{L} U_{R}\right)^{-1}+D_{R}\left(V_{R} V_{L}\right) D_{L}\right.}_{U D V})^{-1} U_{R}^{-1}\\ =&\left(V U_{L}\right)^{-1} D^{-1}\left(U_{R} U^{-1}\right) \end{aligned}$$


Method 3

This method is the one we actually use (divided into using QR decomposition and SVD decomposition (or modified Gram-Schmidt (MGS) factorization), etc.):

SVD (or modified Gram-Schmidt (MGS) factorization)
$$\begin{aligned} G(\tau, \tau) &=[1+B(\tau, 0) B(\beta, \tau)]^{-1} \\ &=\left[1+U_{R} D_{R} V_{R} V_{L} D_{L} U_{L}\right]^{-1} \\ &=U_{L}^{-1}\left[\left(U_{L} U_{R}\right)^{-1}+D_{R}\left(V_{R} V_{L}\right) D_{L}\right]^{-1} U_{R}^{-1} \\ &=U_{L}^{-1}\left[\left(U_{L} U_{R}\right)^{-1}+D_{R}^{\max } D_{R}^{\min }\left(V_{R} V_{L}\right) D_{L}^{\min } D_{L}^{\max }\right]^{-1} U_{R}^{-1} \\ &=U_{L}^{-1}\left(D_{L}^{\max }\right)^{-1}\left[\left(D_{R}^{\max }\right)^{-1}\left(U_{L} U_{R}\right)^{-1}\left(D_{L}^{\max }\right)^{-1}+D_{R}^{\min } V_{R} V_{L} D_{L}^{\min }\right]^{-1}\left(D_{R}^{\max }\right)^{-1} U_{R}^{-1} \end{aligned}$$
The third equality was also suggested by White et al. back in the day.

Where in the above equation: $D_{L}^{\max }(i,i)= \max \left\{ D_{L}(i,i),1 \right\}$, $D_{L}^{\min }(i,i)= \min \left\{ D_{L}(i,i),1 \right\}$

Similarly, we have:
$$\begin{aligned} G(\tau, 0) &=G(\tau, \tau) B(\tau, 0) \\ &=[1+B(\tau, 0) B(\beta, \tau)]^{-1} B(\tau, 0) \\ &=\left[1+U_{R} D_{R} V_{R} V_{L} D_{L} U_{L}\right]^{-1} U_{R} D_{R} V_{R} \\ &=U_{L}^{-1}\left[D_{R}^{-1} U_{R}^{-1} U_{L}^{-1}+V_{R} V_{L} D_{L}\right]^{-1} V_{R} \\ &=U_{L}^{-1}\left[\left(D_{R}^{\max } D_{R}^{\min }\right)^{-1} U_{R}^{-1} U_{L}^{-1}+V_{R} V_{L}\left(D_{L}^{\min } D_{R}^{\max }\right)\right]^{-1} V_{R} \\ &=U_{L}^{-1}\left(D_{L}^{\max }\right)^{-1}\left[\left(D_{R}^{\max }\right)^{-1}\left(U_{L} U_{R}\right)^{-1}\left(D_{L}^{\max }\right)^{-1}+D_{R}^{\min } V_{R} V_{L} D_{L}^{\min }\right]^{-1} D_{R}^{\min } V_{R} \end{aligned}$$
And:
$$\begin{aligned} G(0,0) &=[1+B(\beta, 0)]^{-1} \\ &=[1+B(\beta, \tau) B(\tau, 0)]^{-1} \\ &=\left[1+V_{L} D_{L} U_{L} U_{R} D_{R} V_{R}\right]^{-1} \\ &=V_{R}^{-1}\left[\left(V_{R} V_{L}\right)^{-1}+D_{L}\left(U_{L} U_{R}\right) D_{R}\right]^{-1} V_{L}^{-1} \\ &=V_{R}^{-1}\left[\left(V_{R} V_{L}\right)^{-1}+D_{L}^{\max } D_{L}^{\min }\left(U_{L} U_{R}\right) D_{R}^{\min } D_{R}^{\max }\right]^{-1} V_{L}^{-1} \\ &=V_{R}^{-1}\left(D_{R}^{\max }\right)^{-1}\left[\left(D_{L}^{\max }\right)^{-1}\left(V_{R} V_{L}\right)^{-1}\left(D_{R}^{\max }\right)^{-1}+D_{L}^{\min }\left(U_{L} U_{R}\right) D_{R}^{\min }\right]^{-1}\left(D_{L}^{\max }\right)^{-1} V_{L}^{-1} \end{aligned}$$
Also:
$$\begin{aligned} G(0, \tau) &=-B^{-1}(\tau, 0)(1-G(\tau, \tau)) \\ &=-B^{-1}(\tau, 0) B(\tau, 0)[1+B(\beta, 0)]^{-1} B(\beta, \tau) \\ &=-[1+B(\beta, 0)]^{-1} B(\beta, \tau) \\ &=-\left[1+V_{L} D_{L} U_{L} U_{R} D_{R} V_{R}\right]^{-1} V_{L} D_{L} U_{L} \\ &=-V_{R}^{-1}\left[D_{L}^{-1} V_{L}^{-1} V_{R}^{-1}+U_{L} U_{R} D_{R}\right]^{-1} U_{L} \\ &=-V_{R}^{-1}\left(D_{R}^{\max }\right)^{-1}\left[\left(D_{L}^{\max }\right)^{-1}\left(V_{R} V_{L}\right)^{-1}\left(D_{R}^{\max }\right)^{-1}+D_{L}^{\min }\left(U_{L} U_{R}\right) D_{R}^{\min }\right]^{-1} D_{L}^{\min } U_{L} \end{aligned}$$
QR
$$\begin{aligned} G(\tau, \tau) &=[1+B(\tau, 0) B(\beta, \tau)]^{-1} \\ &=\left[1+Q_{R} D_{R} T_{R} T_{L} D_{L} Q_{L}\right]^{-1} \\ &=Q_{L}^{-1}\left[\left(Q_{L} Q_{R}\right)^{-1}+D_{R} T_{R} T_{L} D_{L}\right]^{-1} Q_{R}^{-1} \\ &=Q_{L}^{-1}\left[\left(Q_{L} Q_{R}\right)^{-1}+D_{R}^{\max } D_{R}^{\min } T_{R} T_{L} D_{L}^{\min } D_{L}^{\max }\right]^{-1} Q_{R}^{-1} \\ &=Q_{L}^{-1}\left(D_{L}^{\max }\right)^{-1}\left[\left(D_{R}^{\max }\right)^{-1}\left(Q_{L} Q_{R}\right)^{-1}\left(D_{L}^{\max }\right)^{-1}+D_{R}^{\min } T_{R} T_{L} D_{L}^{\min }\right]^{-1}\left(D_{R}^{\max }\right)^{-1} Q_{R}^{-1} \end{aligned}$$
Other cases will not be given as examples.

In short, what we hope to do is to separate the large scales and small scales as much as possible, and only mix them together at the very end.


#### General Workflow


In the code, we actually only need one set of UDV variables to exist. When updating to the $\tau$-th slice, we have half $U_{R} D_{R} V_{R}$ and half $V_{L} D_{L} U_{L}$ respectively. Note that the equal-time Green's function requires the UDV matrices of $B(\tau, 0)$ and $B(\beta, \tau)$ on both sides.

For example, initially, obtain all the UDV matrices corresponding to $B(\tau, 0)$. Then, updating the auxiliary field layer by layer starting from the $\beta$-th layer, we only need to gradually obtain the corresponding VDU matrices of $B(\beta, \tau)$ up to the $\tau$-th layer.

#### Sign Problem


We know that when performing simulations of quantum systems, we actually transform them into classical probability distributions to carry out Monte Carlo sampling. In fact, for Determinant Quantum Monte Carlo, the weight corresponding to each auxiliary field configuration is the determinant of a certain matrix. If there are no restrictions, the value of a determinant can naturally be negative or even complex. If the corresponding weight is not a positive real number, then we cannot proceed with Markov Chain Monte Carlo calculations (because the weight ratio is related to our acceptance probability, and with a negative or complex acceptance probability, we cannot continue the calculation).

It is worth noting that the emergence of the sign problem is related to the method of decoupling, and similarly, it is also related to the choice of basis for calculation (obviously, if the energy eigenstates are chosen, there is no sign problem). The sign problem is an NP-hard problem, meaning there is no universal general solution for the sign problem.

Below we briefly introduce two categories where, under certain conditions, there are proofs for the absence of the sign problem:

##### Case 1


If a system has anti-unitary symmetry, then the system has no sign problem.

For the properties of anti-unitary transformations, see the appendix.

The specific expression is, if there exists an anti-unitary transformation $\mathcal{K}$ such that:
$$\begin{aligned} \mathcal{K} T \mathcal{K}^{\dagger} &=T \\ \mathcal{K} V(\mathcal{C}) \mathcal{K}^{\dagger} &=V(\mathcal{C}) \\ \mathcal{K}^{2} &=-1 \end{aligned}$$
Then the eigenvalues of the matrix $1+B_{C}(\beta, 0)$ related to the weight of the fermion part appear in pairs (i.e., if there is an eigenvalue $\lambda_i$, then there is correspondingly another eigenvalue, its conjugate $\lambda_i^{*}$), namely $\det \left( 1+B_{C}(\beta, 0) \right) =\prod_i |\lambda_i|^2$.
Where in the above equations, $T$ and $V(\boldsymbol{s})$ are the kinetic energy term and the interaction term after decoupling, respectively.

For the weight corresponding matrix $1+B_{C}(\beta, 0)$, assuming there is an eigenvalue $\lambda$ and its corresponding eigenvector is $\mathcal{V}$, we have:
$$\left(1+B_{C}(\beta, 0)\right) \mathcal{V}=\lambda \mathcal{V}$$
Because $\mathcal{K}(1+B(\beta, 0)) \mathcal{K}^{\dagger}=(1+B(\beta, 0))$, then:
$$(1+B(\beta, 0)) \mathcal{K} \mathcal{V}=\mathcal{K}(1+B(\beta, 0)) \mathcal{K}^{\dagger} \mathcal{K} \mathcal{V}=\lambda^{*} \mathcal{K} \mathcal{V}$$
Obviously, $\lambda^{*}$ is also its eigenvalue. By the properties of anti-unitary transformations, we have:
$$(\mathcal{V}, \mathcal{K} \mathcal{V})=\langle\mathcal{V}|\mathcal{K}| \mathcal{V}\rangle=\left\langle\mathcal{V}\left|\mathcal{K}^{\dagger} \mathcal{K} \mathcal{K}\right| \mathcal{V}\right\rangle^{*}=-\left\langle\mathcal{V}\left|\mathcal{K}^{\dagger}\right| \mathcal{V}\right\rangle^{*}=-(\mathcal{K} \mathcal{V}, \mathcal{V})^{*}=-(\mathcal{V}, \mathcal{K} \mathcal{V})$$
We know that $\mathcal{V}$ and $\mathcal{K} \mathcal{V}$ are orthogonal, which means $\lambda$ and $\lambda^{*}$ are eigenvalues corresponding to different eigenvectors.

And for a Hamiltonian like the following:
$$\left\{\begin{aligned} H_{\text {fermion }} &=-t \sum_{\langle i j\rangle \sigma \lambda} \hat{c}_{i \lambda \sigma}^{\dagger} \hat{c}_{j \lambda \sigma}+\text {h.c.}-\mu \sum_{i \sigma \lambda} \hat{n}_{i \lambda \sigma} \\ H_{\text {spin}} &=-J \sum_{\langle i j\rangle} \hat{s}_{i}^{z} \hat{s}_{j}^{z}-h_{x} \sum_{i} \hat{s}_{i}^{x} \\ H_{f-s} &=-\xi \sum_{i} s_{i}^{z}\left(\hat{c}_{i}^{\dagger} \tau_{z} \sigma_{z} \hat{c}_{i}\right)=-\xi \sum_{i} s_{i}^{z}\left(\hat{\sigma}_{i 1}^{z}-\hat{\sigma}_{i 2}^{z}\right) \end{aligned}\right.$$
We notice that the up part becomes the down part under the anti-unitary transformation:
$$ \begin{aligned} \hat{h}_{\uparrow} \quad=& \Delta \tau \cdot t \sum_{\langle i j\rangle \lambda} \hat{c}_{i \lambda \uparrow}^{\dagger} \hat{c}_{j \lambda \uparrow}+h . c .+\Delta \tau \cdot \mu \sum_{i \lambda} \hat{n}_{i \lambda \uparrow}+\Delta \tau \cdot \xi \sum_{i} s_{i}^{z}\left(\hat{n}_{i 1 \uparrow}-\hat{n}_{i 2 \uparrow}\right) / 2 \\ \stackrel{\mathcal{T}}{\longrightarrow} & \Delta \tau \cdot t \sum_{\langle i j\rangle \lambda} \hat{c}_{i \lambda \downarrow}^{\dagger} \hat{c}_{j \lambda \downarrow}+\text { h.c. }+\Delta \tau \cdot \mu \sum_{i \lambda} \hat{n}_{i \lambda \downarrow}+\Delta \tau \cdot \xi \sum_{i} s_{i}^{z}\left(\hat{n}_{i 1 \downarrow}-\hat{n}_{i 2 \downarrow}\right) / 2 \\ \stackrel{\tau_{x} \text { on orbital space }}{\longrightarrow} & \Delta \tau \cdot t \sum_{\langle i j\rangle \lambda} \hat{c}_{i \lambda \downarrow}^{\dagger} \hat{c}_{j \lambda \downarrow}+\text { h.c. }+\Delta \tau \cdot \mu \sum_{i \lambda} \hat{n}_{i \lambda \downarrow}+\Delta \tau \cdot \xi \sum_{i} s_{i}^{z}\left(\hat{n}_{i 2 \downarrow}-\hat{n}_{i 1 \downarrow}\right) / 2 \\ & \hat{h}_{\downarrow} \end{aligned} $$
Thus, the weight of the down part is the complex conjugate of the up part, and in practice, one only needs to calculate the weight (ratio) of the up part, and there is no need to calculate the weight (ratio) of the down part anymore.

##### Case 2



For models such as the Hubbard model, the situation is slightly different. For instance, we can only calculate the half-filled case where $U>0$, because it is guaranteed by unitary Particle-Hole transformations like the following (which is well-defined only on a bipartite lattice):
$$\left\{\begin{array}{l} c_{i \downarrow}^{\dagger} \rightarrow(-1)^{i} c_{i \downarrow} \\ c_{i \downarrow} \rightarrow(-1)^{i} c_{i \downarrow}^{\dagger} \end{array}\right.$$
The hopping term remains unchanged, while for the parts decoupled into different channels:
$$\text {$S_z$ Channel $\alpha$ is real} \rightarrow\left\{\begin{array}{c} \alpha\left(n_{i \uparrow}-n_{i \downarrow}\right)\stackrel{P H \text { for spin-down }}{\longrightarrow} \alpha s_{i}\left(n_{i \uparrow}+n_{i \downarrow}-1\right) \\ e^{\alpha s_{i}\left(n_{i \uparrow}-n_{i \downarrow}\right)} \stackrel{P H \text { for spin-down }}{\longrightarrow} e^{\alpha s_{i}\left(n_{i \uparrow}+n_{i \downarrow}-1\right)} \end{array}\right.$$
In the $S_z$ Channel, $\alpha$ is real. Looking solely at the result before the transformation, it is impossible to see whether a negative sign problem exists. But after the Particle-Hole transformation, we see that it is identical for spin up and down. The final weight should be the square of the up part, multiplied by a real $\exp$-type factor, so naturally, there is no negative sign problem (if all matrix elements are real, there is even less chance of a complex sign problem).
$$\text {CDW Channel $\alpha$ is imaginary} \rightarrow\left\{\begin{array}{c} \alpha\left(n_{i \uparrow}+n_{i \downarrow}-1\right)\stackrel{P H \text { for spin-down }}{\longrightarrow} \alpha s_{i}\left(n_{i \uparrow}-n_{i \downarrow}\right) \\ e^{\alpha s_{i}\left(n_{i \uparrow}+n_{i \downarrow}-1\right)} \stackrel{P H \text { for spin-down }}{\longrightarrow} e^{\alpha s_{i}\left(n_{i \uparrow}-n_{i \downarrow}\right)} \end{array}\right.$$
In the CDW Channel, $\alpha$ is imaginary. Looking solely at the result before the transformation, it is impossible to see whether a complex sign problem exists. But after the unitary Particle-Hole transformation, the value of the weight is unchanged. We see that the respective matrix elements for spin up and down are complex conjugates of each other. After the Particle-Hole transformation, the weight is a positive real number, so the weight is naturally a positive real number.